# Exploratory Data Analysis: PSR Shadow Boundary Segmentation

This notebook explores the Chandrayaan-2 OHRC dataset, examining pixel intensity distributions,
class balance, and spatial characteristics of shadow/illuminated regions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
import sys

sys.path.insert(0, str(Path('..').resolve()))
from src.data.splits import get_split_info, print_split_summary

plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150

## 1. Dataset Structure

In [ ]:
with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

patches_dir = Path(config['data']['base_dir']) / 'patches'
print_split_summary(str(patches_dir))

## 2. Intensity Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, cls in zip(axes, ['psr', 'sunlit', 'mixed']):
    data = np.load(patches_dir / cls / 'train.npy')
    means = data.mean(axis=(1, 2))
    stds = data.std(axis=(1, 2))
    
    ax.hist(means, bins=50, alpha=0.7, edgecolor='black')
    ax.set_title(f'{cls.upper()} - Mean Intensity')
    ax.set_xlabel('Mean Intensity')
    ax.set_ylabel('Count')
    ax.axvline(means.mean(), color='r', linestyle='--', label=f'Mean: {means.mean():.4f}')
    ax.legend()

plt.tight_layout()
plt.show()

## 3. Sample Patches Visualization

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(18, 9))

for i, cls in enumerate(['psr', 'sunlit', 'mixed']):
    data = np.load(patches_dir / cls / 'train.npy')
    indices = np.random.choice(len(data), 6, replace=False)
    for j, idx in enumerate(indices):
        axes[i, j].imshow(data[idx], cmap='gray', vmin=0, vmax=1)
        axes[i, j].set_title(f'{cls} #{idx}\nstd={data[idx].std():.3f}')
        axes[i, j].axis('off')

plt.suptitle('Sample Patches by Class', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Class Balance

In [ ]:
info = get_split_info(str(patches_dir))

counts = {}
for cls, splits in info.items():
    counts[cls] = sum(s['count'] for s in splits.values())

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(counts.keys(), counts.values(), color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax.set_ylabel('Number of Patches')
ax.set_title('Class Distribution')
for bar, count in zip(bars, counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{count:,}', ha='center', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Statistics Summary

In [ ]:
print("\n" + "="*60)
print("DATASET STATISTICS SUMMARY")
print("="*60)

for cls in ['psr', 'sunlit', 'mixed']:
    train_data = np.load(patches_dir / cls / 'train.npy')
    val_data = np.load(patches_dir / cls / 'val.npy')
    
    print(f"\n{cls.upper()}:")
    print(f"  Train: {len(train_data):,} patches, mean={train_data.mean():.4f}, std={train_data.std():.4f}")
    print(f"  Val:   {len(val_data):,} patches, mean={val_data.mean():.4f}, std={val_data.std():.4f}")
    print(f"  Pixel range: [{train_data.min():.4f}, {train_data.max():.4f}]")
    print(f"  % pixels < 0.05: {(train_data < 0.05).mean():.2%}")